# Lab 8.2: SHAP for a Credit Card Default Model

Run the cells from top to bottom. You do not need to change any code. Each checkpoint tells you what to look for. This notebook reproduces the case study of Chapter 8.

Data: Default of Credit Card Clients (Yeh and Lien, 2009), UCI Machine Learning Repository, DOI 10.24432/C55S3H, CC BY 4.0. 30,000 credit card customers in Taiwan, April to September 2005. Target: default on the next month's payment.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import pathlib, tempfile, urllib.request, zipfile
import numpy as np, pandas as pd, shap, sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
print("numpy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__, "| shap", shap.__version__)

URL = "https://archive.ics.uci.edu/static/public/350/default+of+credit+card+clients.zip"
here = pathlib.Path("default of credit card clients.xls")
tmp = pathlib.Path(tempfile.gettempdir()) / "xai-ch08"; tmp.mkdir(exist_ok=True)
xls = here if here.exists() else tmp / here.name
if not xls.exists():
    urllib.request.urlretrieve(URL, tmp / "cc.zip")
    zipfile.ZipFile(tmp / "cc.zip").extractall(tmp)
d = pd.read_excel(xls, header=1)
print(d.shape, "default rate", round(d["default payment next month"].mean(), 4))

numpy 2.2.6 | pandas 2.3.3 | scikit-learn 1.7.2 | shap 0.49.1


(30000, 25) default rate 0.2212


**Checkpoint 1:** 30,000 rows and 25 columns; a default rate of 0.2212.

## 1. The model

We use 19 features: the credit limit, six monthly repayment statuses (`PAY_0` is September, the latest; `PAY_6` is April), six bill amounts and six payment amounts. Sex, education, marital status and age are left out on purpose.

In [2]:
F = (["LIMIT_BAL", "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
     + [f"BILL_AMT{i}" for i in range(1, 7)] + [f"PAY_AMT{i}" for i in range(1, 7)])
X = d[F].astype(float); y = d["default payment next month"].values
idx = np.arange(len(d))
Xtr, Xte, ytr, yte, itr, ite = train_test_split(X, y, idx, test_size=0.25, stratify=y, random_state=0)
m = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
                                   early_stopping=False, random_state=0).fit(Xtr, ytr)
p = m.predict_proba(Xte)[:, 1]
thr = np.quantile(p, 0.8); flag = p >= thr
print("test AUC", round(roc_auc_score(yte, p), 3))
print("flag threshold", round(thr, 3), "| flagged", flag.sum(), "| default rate among flagged", round(yte[flag].mean(), 3))
print("PAY_0 values:", d["PAY_0"].value_counts().sort_index().to_dict())

test AUC 0.775
flag threshold 0.332 | flagged 1500 | default rate among flagged 0.562
PAY_0 values: {-2: 2759, -1: 5686, 0: 14737, 1: 3688, 2: 2667, 3: 322, 4: 76, 5: 26, 6: 11, 7: 9, 8: 19}


**Checkpoint 2:** test AUC 0.775; threshold 0.332; 1,500 flagged customers, of whom 0.562 defaulted. Note that `PAY_0` takes the values -2 and 0, which the dataset documentation does not define (it documents -1 = paid duly and 1 to 9 = months of delay).

## 2. Explaining one customer: Mei

Interventional TreeSHAP against a random background of 200 training customers. The values are in log-odds and add up to the gap between Mei's log-odds and the base value.

In [3]:
sig = lambda z: 1 / (1 + np.exp(-z))
bgA = Xtr.sample(200, random_state=0)
exA = shap.TreeExplainer(m, data=bgA, feature_perturbation="interventional")
def reasons(ex, row, k=6):
    s = ex.shap_values(Xte.iloc[[row]])[0]; b = float(ex.expected_value)
    o = np.argsort(-np.abs(s))[:k]
    t = pd.DataFrame({"feature": [F[j] for j in o], "value": [Xte.iloc[row][F[j]] for j in o], "shap": s[o].round(3)})
    return b, s, t
mei = 7
b, s, t = reasons(exA, mei)
print(f"base {b:.3f} (risk {sig(b):.1%}) -> Mei {b + s.sum():.3f} (risk {p[mei]:.1%}); sum of SHAP values {s.sum():.3f}")
t

base -1.484 (risk 18.5%) -> Mei 0.560 (risk 63.7%); sum of SHAP values 2.044


,feature,value,shap
0,PAY_0,2.0,1.932
1,LIMIT_BAL,50000.0,0.193
2,BILL_AMT1,49316.0,-0.074
3,PAY_AMT4,431.0,-0.043
4,PAY_6,0.0,-0.033
5,PAY_5,0.0,-0.032


**Checkpoint 3:** base -1.484 (risk 18.5 percent), Mei 0.560 (risk 63.7 percent). `PAY_0` (two months late in September) contributes +1.932; the credit limit +0.193; the September bill -0.074.

## 3. A SHAP value is not the effect of a change

What happens if we set Mei's September status to -1 (paid duly) and keep everything else?

In [4]:
x = Xte.iloc[[mei]].copy(); x2 = x.copy(); x2["PAY_0"] = -1
print("log-odds change", round(float(m.decision_function(x2)[0] - m.decision_function(x)[0]), 3),
      "| risk", round(float(m.predict_proba(x)[0, 1]), 3), "->", round(float(m.predict_proba(x2)[0, 1]), 3))

log-odds change -2.395 | risk 0.637 -> 0.138


**Checkpoint 4:** the log-odds fall by 2.395 and the risk from 0.637 to 0.138: not the same number as the SHAP value of +1.932, because the two numbers answer different questions.

## 4. The background is the foil: Daniel

In [5]:
bgB = Xtr[ytr == 0].sample(200, random_state=0)
exB = shap.TreeExplainer(m, data=bgB, feature_perturbation="interventional")
exC = shap.TreeExplainer(m, data=Xtr.median().to_frame().T, feature_perturbation="interventional")
dan = 75
for name, ex in [("A: all customers", exA), ("B: customers who repaid", exB), ("C: one median customer", exC)]:
    b, s, t = reasons(ex, dan, 4)
    print(f"{name:26s} base risk {sig(b):.1%} | top reasons:", list(zip(t.feature, t.shap)))

A: all customers           base risk 18.5% | top reasons: [('PAY_2', 0.256), ('PAY_4', 0.232), ('PAY_3', 0.216), ('PAY_0', 0.168)]
B: customers who repaid    base risk 14.1% | top reasons: [('PAY_0', 0.333), ('PAY_2', 0.289), ('PAY_4', 0.262), ('PAY_3', 0.241)]
C: one median customer     base risk 8.9% | top reasons: [('PAY_0', 0.49), ('PAY_4', 0.362), ('PAY_3', 0.27), ('PAY_2', 0.268)]


**Checkpoint 5:** against all customers Daniel's top reason is `PAY_2` (August); against customers who repaid it is `PAY_0` (September); against one median customer `PAY_0` again, with a base risk of 8.9 percent.

## 5. How often does the background change the reasons?

For 300 randomly chosen flagged customers, compare the top risk-raising reason and the top three under backgrounds A and B, under path-dependent TreeSHAP, and across five random backgrounds of the same kind as A.

In [6]:
fl = np.where(flag)[0]; sub = np.random.RandomState(1).choice(fl, 300, replace=False)
top3 = lambda S: [set(np.argsort(-r)[:3]) for r in S]
jac = lambda u, v: np.mean([len(a & b) / len(a | b) for a, b in zip(u, v)])
SA = exA.shap_values(Xte.iloc[sub]); SB = exB.shap_values(Xte.iloc[sub])
SP = shap.TreeExplainer(m, feature_perturbation="tree_path_dependent").shap_values(Xte.iloc[sub])
print("top reason differs A vs B:", round(np.mean(SA.argmax(1) != SB.argmax(1)), 3))
print("mean Jaccard of top three, A vs B:", round(jac(top3(SA), top3(SB)), 3), "| A vs path-dependent:", round(jac(top3(SA), top3(SP)), 3))
T = [top3(shap.TreeExplainer(m, data=Xtr.sample(200, random_state=100 + k), feature_perturbation="interventional").shap_values(Xte.iloc[sub])) for k in range(5)]
same = [np.mean([a == b for a, b in zip(T[i], T[j])]) for i in range(5) for j in range(i + 1, 5)]
print("five random backgrounds: identical top-three sets in", round(np.mean(same), 3), "of pairs")

top reason differs A vs B: 0.153
mean Jaccard of top three, A vs B: 0.842 | A vs path-dependent: 0.792


five random backgrounds: identical top-three sets in 0.739 of pairs


**Checkpoint 6:** the top reason differs for 0.153 of these customers; Jaccard 0.842 (A vs B) and 0.792 (A vs path-dependent); identical top-three sets in 0.739 of pairs of random backgrounds.

## 6. Global view and grouped features

In [7]:
S = exA.shap_values(Xte.iloc[:2000])
imp = pd.Series(np.abs(S).mean(0), index=F).sort_values(ascending=False)
groups = {"payment status": F[1:7], "bill amounts": F[7:13], "payments made": F[13:19], "credit limit": ["LIMIT_BAL"]}
gimp = {g: round(np.abs(S[:, [F.index(f) for f in fs]].sum(1)).mean(), 3) for g, fs in groups.items()}
print(imp.head(8).round(3).to_dict()); print(gimp)
print("bill amounts correlate", round(d[F[7:13]].corr().values[np.triu_indices(6, 1)].min(), 2), "to", round(d[F[7:13]].corr().values[np.triu_indices(6, 1)].max(), 2))

{'PAY_0': 0.431, 'LIMIT_BAL': 0.198, 'BILL_AMT1': 0.164, 'PAY_AMT2': 0.1, 'PAY_2': 0.1, 'PAY_AMT1': 0.097, 'PAY_AMT3': 0.09, 'PAY_3': 0.075}
{'payment status': np.float64(0.684), 'bill amounts': np.float64(0.191), 'payments made': np.float64(0.298), 'credit limit': np.float64(0.198)}
bill amounts correlate 0.8 to 0.95


**Checkpoint 7:** `PAY_0` leads (0.431), then `LIMIT_BAL` (0.198) and `BILL_AMT1` (0.164). Grouped: payment status 0.684, payments made 0.298, credit limit 0.198, bill amounts 0.191. Bill amounts correlate 0.80 to 0.95.

## 7. What SHAP does not show: fairness

In [8]:
sex = d.loc[ite, "SEX"].values  # 1 = male, 2 = female
print("flag rate men", round(flag[sex == 1].mean(), 3), "| women", round(flag[sex == 2].mean(), 3))
print("default rate men", round(yte[sex == 1].mean(), 3), "| women", round(yte[sex == 2].mean(), 3))

flag rate men 0.215 | women 0.19
default rate men 0.244 | women 0.206


**Checkpoint 8:** flag rates 0.215 (men) and 0.190 (women); default rates 0.244 and 0.206. Sex is not an input, so its SHAP value is zero for everyone, yet the flag rates differ.

## Your hand-in

Pick five flagged customers (use `fl[:5]` or your own choice). For each, list the top three reasons under background A and under background B, and note whether the top reason changed. Then write one paragraph: which background would you use for a customer notice, and why?

In [9]:
for r in fl[:5]:
    a = [F[j] for j in np.argsort(-exA.shap_values(Xte.iloc[[r]])[0])[:3]]
    b_ = [F[j] for j in np.argsort(-exB.shap_values(Xte.iloc[[r]])[0])[:3]]
    print(r, "risk", round(p[r], 3), "| A:", a, "| B:", b_, "| top changed:", a[0] != b_[0])

7 risk 0.637 | A: ['PAY_0', 'LIMIT_BAL', 'PAY_AMT1'] | B: ['PAY_0', 'LIMIT_BAL', 'BILL_AMT3'] | top changed: False
12 risk 0.391 | A: ['PAY_6', 'PAY_3', 'PAY_4'] | B: ['PAY_6', 'PAY_3', 'PAY_4'] | top changed: False
14 risk 0.558 | A: ['PAY_0', 'LIMIT_BAL', 'PAY_AMT3'] | B: ['PAY_0', 'LIMIT_BAL', 'PAY_AMT3'] | top changed: False
17 risk 0.605 | A: ['PAY_0', 'PAY_AMT4', 'LIMIT_BAL'] | B: ['PAY_0', 'LIMIT_BAL', 'PAY_AMT4'] | top changed: False
19 risk 0.612 | A: ['PAY_0', 'LIMIT_BAL', 'PAY_AMT6'] | B: ['PAY_0', 'LIMIT_BAL', 'BILL_AMT3'] | top changed: False
